<a href="https://colab.research.google.com/github/Kauustubbh/Skin-Lesion/blob/main/notebooks/02_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import drive, userdata

# 1. Ensure Drive is mounted
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/SkinLesionClassifier'

# 2. Load token from Colab Secrets
try:
    token = userdata.get('Kaggle_api_key')
except:
    token = userdata.get('KAGGLE_API_TOKEN')

# Set environment variable for Kaggle CLI
os.environ['KAGGLE_API_TOKEN'] = token

# 3. Download and unzip dataset
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 --unzip -p data/
print("✅ Dataset downloaded and unzipped successfully.")

In [ ]:
import pandas as pd
import glob

# 1. Read split contract exported by Notebook 01 from Drive
split_csv_path = os.path.join(PROJECT_DIR, 'train_val_test_split.csv')
df = pd.read_csv(split_csv_path)

# 2. Resolve image paths across both downloaded subfolders
img_paths = glob.glob('data/HAM10000_images_part_1/*.jpg') + glob.glob('data/HAM10000_images_part_2/*.jpg')
img_lookup = {os.path.splitext(os.path.basename(p))[0]: p for p in img_paths}

df['path'] = df['image_id'].map(img_lookup)

# 3. Encoded diagnostic class shortcodes to 0-6 integer labels
label_mapping = {'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3, 'mel': 4, 'nv': 5, 'vasc': 6}
df['label'] = df['dx'].map(label_mapping)

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df = df[df['split'] == 'val'].reset_index(drop=True)

print(f"✅ Split contract loaded successfully: {len(train_df)} train samples, {len(val_df)} validation samples.")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# Geometric augmentations ONLY (preserves clinical skin lesion color features)
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=90),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class SkinDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        label = row['label']
        if self.transform:
            image = self.transform(image)
        return image, label

BATCH_SIZE = 64  # Increased batch size for fast T4 GPU processing

train_loader = DataLoader(
    SkinDataset(train_df, train_transforms),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    SkinDataset(val_df, val_transforms),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Compute inverse frequency class weights to fix dataset imbalance
class_counts = train_df['label'].value_counts().sort_index().values
total_samples = len(train_df)
num_classes = len(class_counts)

class_weights = total_samples / (num_classes * class_counts)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class_weights_tensor = class_weights_tensor.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
print("✅ Datasets and fast DataLoaders ready. Device set to:", device)

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet50, ResNet50_Weights
from torch.cuda.amp import autocast, GradScaler

num_classes = 7
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Initialize ResNet-50 with pretrained ImageNet weights
model = resnet50(weights=ResNet50_Weights.DEFAULT)

# 2. Freeze all backbone layers initially
for param in model.parameters():
    param.requires_grad = False

# Replace final head (2048 in_features -> 7 diagnostic classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

scaler = GradScaler()
checkpoint_path = os.path.join(PROJECT_DIR, 'baseline_resnet50.pth')
best_val_loss = float('inf')

# ====================================================================
# STAGE 1: Classifier Head Warmup (3 Epochs)
# ====================================================================
optimizer_stage1 = optim.Adam(model.fc.parameters(), lr=1e-3)
print("🚀 Starting Stage 1: FC Head Warmup (3 Epochs)...")

for epoch in range(1, 4):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer_stage1.zero_grad()

        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer_stage1)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    print(f"Stage 1 - Epoch {epoch}/3 | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.3f}")

# ====================================================================
# STAGE 2: Fine-Tuning Deep Layers (Layer 4 + FC Head) (10 Epochs)
# ====================================================================
for param in model.layer4.parameters():
    param.requires_grad = True

# Differential learning rates: gentle nudge on layer4, higher on fc head
optimizer_stage2 = optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.fc.parameters(),     'lr': 1e-4}
])

print("\n🚀 Starting Stage 2: Fine-Tuning Layer 4 & FC Head (10 Epochs)...")
num_epochs_stage2 = 10

for epoch in range(1, num_epochs_stage2 + 1):
    # --- Training Loop ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer_stage2.zero_grad()

        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer_stage2)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # --- Validation Loop ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss / val_total
    val_acc = val_correct / val_total

    print(f"Epoch {epoch:02d}/{num_epochs_stage2} | Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.3f}")

    # Checkpoint: Save model weights ONLY when validation loss improves
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), checkpoint_path)
        print(f"   --> 💾 Saved Best Model Checkpoint (Val Loss: {val_loss:.4f}) to Drive!")

print(f"\n🎉 Training complete! Best model checkpoint verified at: {checkpoint_path}")